# SVR (Support Vector Regressor) - Network Throughput Prediction

Target: `Data Throughput (Mbps)`

ملحوظة: SVR بيتأثر بمقياس الأرقام، فبنستخدم StandardScaler للأعمدة الرقمية (على عكس XGBoost وRandom Forest).

## 1. Clone repo (Colab only)

In [1]:
!git clone https://github.com/MohamedAAbdullah1/NTI-Final-Project.git
%cd NTI-Final-Project

Cloning into 'NTI-Final-Project'...
remote: Enumerating objects: 105, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 105 (delta 45), reused 43 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (105/105), 3.26 MiB | 6.56 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/NTI-Final-Project


## 2. Imports

In [3]:
import os
import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## 3. Load data

In [4]:
X_train = pd.read_csv("data/processed/X_train.csv")
X_test = pd.read_csv("data/processed/X_test.csv")
y_train = pd.read_csv("data/processed/y_train.csv").values.ravel()
y_test = pd.read_csv("data/processed/y_test.csv").values.ravel()

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test:", X_test.shape, "| y_test:", y_test.shape)
X_train.head()

X_train: (13463, 6) | y_train: (13463,)
X_test: (3366, 6) | y_test: (3366,)


,Locality,Latitude,Longitude,Signal Strength (dBm),Latency (ms),Network Type
0,Pataliputra,25.605539,85.285651,-87.611830,164.101054,4G
1,Kidwaipuri,25.491973,85.086669,-90.089704,22.851745,5G
2,Danapur,25.684972,84.989785,-90.175391,139.812003,4G
3,Kankarbagh,25.569868,85.138714,-93.384826,113.904981,4G
4,Phulwari Sharif,25.626811,85.136823,-95.888704,71.291449,4G


## 4. Preprocessing (with scaling for SVR)

In [5]:
numeric_cols = ["Latitude", "Longitude", "Signal Strength (dBm)", "Latency (ms)"]
categorical_cols = ["Locality", "Network Type"]

svm_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols)
    ]
)

## 5. Build and train the model




In [6]:
svm_model = Pipeline(steps=[
    ("preprocessor", svm_preprocessor),
    ("model", SVR(
        kernel="rbf",
        C=10,
        epsilon=0.5
    ))
])

svm_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Locality', 'Network Type']),
                                                 ('num', StandardScaler(),
                                                  ['Latitude', 'Longitude',
                                                   'Signal Strength (dBm)',
                                                   'Latency (ms)'])])),
                ('model', SVR(C=10, epsilon=0.5))])

## 6. Evaluate

In [7]:
y_pred_svm = svm_model.predict(X_test)

rmse_svm = np.sqrt(mean_squared_error(y_test, y_pred_svm))
mae_svm = mean_absolute_error(y_test, y_pred_svm)
r2_svm = r2_score(y_test, y_pred_svm)

print("RMSE:", rmse_svm)
print("MAE:", mae_svm)
print("R\u00b2:", r2_svm)

RMSE: 13.359680001365053
MAE: 6.594985898528747
R²: 0.7346731081531699


## 7. Save the model

In [8]:
import joblib

os.makedirs("models", exist_ok=True)
joblib.dump(svm_model, "models/svm_model.joblib")
print("Saved to models/svm_model.joblib")

Saved to models/svm_model.joblib
